# 01 · SQL basics (Spark)

SELECT, WHERE, ORDER BY, DISTINCT, CASE, LIMIT, and **NULL** (a missing value).

Remember: `NULL = NULL` is not true. Use `IS NULL` / `IS NOT NULL`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/01_basics.ipynb`](../../spark/practice/01_basics.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. High earners in Engineering  ·  🟢 easy

Show `name` and `salary` of employees in department 1 (Engineering) who earn **more than** 100,000. Highest salary first.

*Expected result: 5 rows · columns: `name`, `salary`*

**How it works:** 1) `WHERE` keeps rows where **both** conditions are true (`AND`). 2) `ORDER BY salary DESC` puts the highest first; `name` breaks ties so the order is always the same.

In [ ]:
%%sql
SELECT name, salary
FROM employees
WHERE dept_id = 1
  AND salary > 100000
ORDER BY salary DESC, name

,name,salary
0,Alice Johnson,250000.00
1,Bob Smith,150000.00
2,Carol White,120000.00
3,David Brown,120000.00
4,Eva Green,110000.00


**DataFrame API:**

In [ ]:
res = (employees
       .filter((F.col("dept_id") == 1) & (F.col("salary") > 100000))
       .select("name", "salary")
       .orderBy(F.desc("salary"), "name"))
res.show()

+-------------+---------+
|         name|   salary|
+-------------+---------+
|Alice Johnson|250000.00|
|    Bob Smith|150000.00|
|  Carol White|120000.00|
|  David Brown|120000.00|
|    Eva Green|110000.00|
+-------------+---------+

**Why / traps:** In the DataFrame API use `&` and `|` (not `and`/`or`) and put each condition in brackets.

## Q2. Cities we sell to  ·  🟢 easy

List each customer `city` **once**, A to Z. Leave out the missing (NULL) city.

*Expected result: 8 rows · columns: `city`*

**How it works:** 1) Filter out the NULL city first. 2) `DISTINCT` keeps one copy of each city. 3) Sort A→Z.

In [ ]:
%%sql
SELECT DISTINCT city
FROM customers
WHERE city IS NOT NULL
ORDER BY city

,city
0,Austin
1,Boston
2,Chicago
3,Denver
4,Houston
5,New York
6,Phoenix
7,Seattle


**DataFrame API:**

In [ ]:
res = (customers
       .filter(F.col("city").isNotNull())
       .select("city").distinct()
       .orderBy("city"))
res.show()

+--------+
|    city|
+--------+
|  Austin|
|  Boston|
| Chicago|
|  Denver|
| Houston|
|New York|
| Phoenix|
| Seattle|
+--------+

**Why / traps:** `DISTINCT` keeps NULL as one value, so filter it out yourself.

## Q3. Top bosses  ·  🟢 easy

Show the `emp_id` and `name` of employees who have **no manager**.

*Expected result: 2 rows · columns: `emp_id`, `name`*

**How it works:** Rows with no manager have `manager_id` = NULL. `IS NULL` is the only test that finds them.

In [ ]:
%%sql
SELECT emp_id, name
FROM employees
WHERE manager_id IS NULL

,emp_id,name
0,1,Alice Johnson
1,29,Chloe Collins


**DataFrame API:**

In [ ]:
res = employees.filter(F.col("manager_id").isNull()).select("emp_id", "name")
res.show()

+------+-------------+
|emp_id|         name|
+------+-------------+
|     1|Alice Johnson|
|    29|Chloe Collins|
+------+-------------+

**Why / traps:** `WHERE manager_id = NULL` returns **0 rows**: any comparison with NULL is unknown, not true.

## Q4. Mid-price electronics and books  ·  🟢 easy

Show `product_name`, `category`, `price` for products in category Electronics or Books with price **from 20 to 50** (both included). Cheapest first, then by name.

*Expected result: 6 rows · columns: `product_name`, `category`, `price`*

**How it works:** `IN (...)` = equal to any value in the list. `BETWEEN 20 AND 50` = `>= 20 AND <= 50`. Both conditions must be true.

In [ ]:
%%sql
SELECT product_name, category, price
FROM products
WHERE category IN ('Electronics', 'Books')
  AND price BETWEEN 20 AND 50
ORDER BY price, product_name

,product_name,category,price
0,Wireless Mouse,Electronics,25.00
1,SQL Basics,Books,30.00
2,Statistics Made Easy,Books,30.00
3,Python Cookbook,Books,40.00
4,Data Engineering 101,Books,45.00
5,Laptop Stand,Electronics,45.00


**DataFrame API:**

In [ ]:
res = (products
       .filter(F.col("category").isin("Electronics", "Books")
               & F.col("price").between(20, 50))
       .select("product_name", "category", "price")
       .orderBy("price", "product_name"))
res.show()

+--------------------+-----------+-----+
|        product_name|   category|price|
+--------------------+-----------+-----+
|      Wireless Mouse|Electronics|25.00|
|          SQL Basics|      Books|30.00|
|Statistics Made Easy|      Books|30.00|
|     Python Cookbook|      Books|40.00|
|Data Engineering 101|      Books|45.00|
|        Laptop Stand|Electronics|45.00|
+--------------------+-----------+-----+

**Why / traps:** `BETWEEN` includes both ends.

## Q5. Names that start with A  ·  🟢 easy

Show `customer_id` and `name` of customers whose name starts with the letter **A**.

*Expected result: 4 rows · columns: `customer_id`, `name`*

**How it works:** `LIKE 'A%'` = starts with A, then anything. In the DataFrame API `startswith` does the same.

In [ ]:
%%sql
SELECT customer_id, name
FROM customers
WHERE name LIKE 'A%'

,customer_id,name
0,1,Aarav Nguyen
1,12,Aisha Silva
2,21,Aarav Nguyen
3,32,Aisha Silva


**DataFrame API:**

In [ ]:
res = customers.filter(F.col("name").startswith("A")).select("customer_id", "name")
res.show()

+-----------+------------+
|customer_id|        name|
+-----------+------------+
|          1|Aarav Nguyen|
|         12| Aisha Silva|
|         21|Aarav Nguyen|
|         32| Aisha Silva|
+-----------+------------+

**Why / traps:** `%` = any number of characters, `_` = exactly one character. LIKE is case-sensitive in BigQuery, Oracle and Spark.

## Q6. Salary bands  ·  🟢 easy

For every employee show `name`, `salary` and `band`:
- `high` when salary is 150,000 or more
- `mid` when 90,000 or more
- otherwise `low`

*Expected result: 30 rows · columns: `name`, `salary`, `band`*

**How it works:** `CASE` tests each `WHEN` from top to bottom and returns the first match; `ELSE` catches the rest. Because 150k is tested first, the 90k rule only sees salaries below 150k.

In [ ]:
%%sql
SELECT name, salary,
       CASE WHEN salary >= 150000 THEN 'high'
            WHEN salary >= 90000  THEN 'mid'
            ELSE 'low'
       END AS band
FROM employees

,name,salary,band
0,Alice Johnson,250000.00,high
1,Bob Smith,150000.00,high
2,Carol White,120000.00,mid
3,David Brown,120000.00,mid
4,Eva Green,110000.00,mid
5,Frank Moore,95000.00,mid
6,Grace Lee,95000.00,mid
7,Henry Clark,140000.00,mid
8,Irene Lewis,90000.00,mid
9,Jack Walker,90000.00,mid


**DataFrame API:**

In [ ]:
res = employees.select(
    "name", "salary",
    F.when(F.col("salary") >= 150000, "high")
     .when(F.col("salary") >= 90000, "mid")
     .otherwise("low").alias("band"))
res.show()

+-------------+---------+----+
|         name|   salary|band|
+-------------+---------+----+
|Alice Johnson|250000.00|high|
|    Bob Smith|150000.00|high|
|  Carol White|120000.00| mid|
|  David Brown|120000.00| mid|
|    Eva Green|110000.00| mid|
|  Frank Moore| 95000.00| mid|
|    Grace Lee| 95000.00| mid|
|  Henry Clark|140000.00| mid|
|  Irene Lewis| 90000.00| mid|
|  Jack Walker| 90000.00| mid|
|   Karen Hall| 90000.00| mid|
|    Leo Allen| 70000.00| low|
|    Mia Young| 65000.00| low|
|  Nathan King|130000.00| mid|
| Olivia Scott| 85000.00| low|
|   Paul Adams| 85000.00| low|
|  Quinn Baker| 60000.00| low|
|Rachel Nelson|135000.00| mid|
|   Sam Carter|100000.00| mid|
|Tina Mitchell| 98000.00| mid|
+-------------+---------+----+
only showing top 20 rows

**Why / traps:** CASE checks the conditions from top to bottom and stops at the first true one, so the order of the WHEN lines matters.

## Q7. Missing amounts  ·  🟢 easy

Show `order_id`, `status` and `amount` for orders whose amount is missing. Show the amount as **0** instead of NULL.

*Expected result: 6 rows · columns: `order_id`, `status`, `amount`*

**How it works:** 1) `WHERE amount IS NULL` finds the missing amounts. 2) `COALESCE(amount, 0)` shows 0 in place of NULL (it returns the first non-NULL value).

In [ ]:
%%sql
SELECT order_id, status, COALESCE(amount, 0) AS amount
FROM orders
WHERE amount IS NULL

,order_id,status,amount
0,1008,completed,0.00
1,1040,completed,0.00
2,1055,cancelled,0.00
3,1073,completed,0.00
4,1079,completed,0.00
5,1080,completed,0.00


**DataFrame API:**

In [ ]:
res = (orders.filter(F.col("amount").isNull())
       .select("order_id", "status", F.coalesce("amount", F.lit(0)).alias("amount")))
res.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|    1008|completed|  0.00|
|    1040|completed|  0.00|
|    1055|cancelled|  0.00|
|    1073|completed|  0.00|
|    1079|completed|  0.00|
|    1080|completed|  0.00|
+--------+---------+------+

**Why / traps:** `COALESCE(a, b)` returns the first value that is not NULL. It works in all 3 engines. Engine versions: BigQuery `IFNULL`, Oracle `NVL`, Spark `nvl`.

## Q8. Top 5 salaries  ·  🟢 easy

Show the 5 highest-paid employees: `name`, `salary`.

*Expected result: 5 rows · columns: `name`, `salary`*

**How it works:** Sort by salary high→low, then keep the first 5 rows (`LIMIT 5`; Oracle `FETCH FIRST 5 ROWS ONLY`).

In [ ]:
%%sql
SELECT name, salary
FROM employees
ORDER BY salary DESC
LIMIT 5

,name,salary
0,Alice Johnson,250000.00
1,Bob Smith,150000.00
2,Henry Clark,140000.00
3,Rachel Nelson,135000.00
4,Nathan King,130000.00


**DataFrame API:**

In [ ]:
res = employees.select("name", "salary").orderBy(F.desc("salary")).limit(5)
res.show()

+-------------+---------+
|         name|   salary|
+-------------+---------+
|Alice Johnson|250000.00|
|    Bob Smith|150000.00|
|  Henry Clark|140000.00|
|Rachel Nelson|135000.00|
|  Nathan King|130000.00|
+-------------+---------+

**Why / traps:** If the 5th and 6th salaries were equal, LIMIT would cut one of them at random. For ties use a window function (topic 05).

## Q9. Page 2 of the product list  ·  🟡 medium

Products sorted by `price` (highest first), then `product_id`. Show **page 2** with 5 products per page (rows 6 to 10): `product_id`, `product_name`, `price`.

*Expected result: 5 rows · columns: `product_id`, `product_name`, `price`*

**How it works:** Sort with a unique tie-breaker, skip the first 5 rows (`OFFSET 5`), then take the next 5. Page n = `OFFSET (n-1)*size`.

In [ ]:
%%sql
SELECT product_id, product_name, price
FROM products
ORDER BY price DESC, product_id
LIMIT 5 OFFSET 5

,product_id,product_name,price
0,4,Laptop Stand,45.00
1,8,Data Engineering 101,45.00
2,9,Python Cookbook,40.00
3,12,Desk Lamp,35.00
4,7,SQL Basics,30.00


**DataFrame API:**

In [ ]:
res = (products.select("product_id", "product_name", "price")
       .orderBy(F.desc("price"), "product_id")
       .offset(5).limit(5))
res.show()

+----------+--------------------+-----+
|product_id|        product_name|price|
+----------+--------------------+-----+
|         4|        Laptop Stand|45.00|
|         8|Data Engineering 101|45.00|
|         9|     Python Cookbook|40.00|
|        12|           Desk Lamp|35.00|
|         7|          SQL Basics|30.00|
+----------+--------------------+-----+

**Why / traps:** Always add a unique column (here `product_id`) to ORDER BY when paging. Without it, rows with the same price can move between pages.

## Q10. Everyone outside Engineering (NULL trap)  ·  🟡 medium

Show `emp_id`, `name`, `dept_id` of employees who are **not** in department 1. The employee with no department (NULL) must be in the result too.

*Expected result: 23 rows · columns: `emp_id`, `name`, `dept_id`*

**How it works:** `dept_id <> 1` is *unknown* for NULL, so that row would be lost. Adding `OR dept_id IS NULL` brings it back.

In [ ]:
%%sql
SELECT emp_id, name, dept_id
FROM employees
WHERE dept_id <> 1
   OR dept_id IS NULL

,emp_id,name,dept_id
0,8,Henry Clark,2.0
1,9,Irene Lewis,2.0
2,10,Jack Walker,2.0
3,11,Karen Hall,2.0
4,12,Leo Allen,2.0
5,13,Mia Young,2.0
6,14,Nathan King,3.0
7,15,Olivia Scott,3.0
8,16,Paul Adams,3.0
9,17,Quinn Baker,3.0


**DataFrame API:**

In [ ]:
res = (employees
       .filter((F.col("dept_id") != 1) | F.col("dept_id").isNull())
       .select("emp_id", "name", "dept_id"))
res.show()

+------+---------------+-------+
|emp_id|           name|dept_id|
+------+---------------+-------+
|     8|    Henry Clark|      2|
|     9|    Irene Lewis|      2|
|    10|    Jack Walker|      2|
|    11|     Karen Hall|      2|
|    12|      Leo Allen|      2|
|    13|      Mia Young|      2|
|    14|    Nathan King|      3|
|    15|   Olivia Scott|      3|
|    16|     Paul Adams|      3|
|    17|    Quinn Baker|      3|
|    18|  Rachel Nelson|      4|
|    19|     Sam Carter|      4|
|    20|  Tina Mitchell|      4|
|    21|      Uma Perez|      4|
|    22| Victor Roberts|      5|
|    23|   Wendy Turner|      5|
|    24|Xavier Phillips|      5|
|    25|  Yara Campbell|      6|
|    26|    Zane Parker|      6|
|    27|      Amy Evans|      6|
+------+---------------+-------+
only showing top 20 rows

**Why / traps:** `dept_id <> 1` alone skips the NULL row: `NULL <> 1` is unknown, and WHERE keeps only true. Interviewers love this trap. Spark also has a null-safe `<=>` / `eqNullSafe`.

## Q11. Label with a default  ·  🟢 easy

For customers 1 to 5 show `customer_id` and one text column `label` like `Aarav Nguyen (Chicago)`. If the city is NULL write `unknown` instead.

*Expected result: 5 rows · columns: `customer_id`, `label`*

**How it works:** Join text pieces with `||`. `COALESCE(city, 'unknown')` replaces a NULL city before joining, so the label is never NULL.

In [ ]:
%%sql
SELECT customer_id,
       name || ' (' || COALESCE(city, 'unknown') || ')' AS label
FROM customers
WHERE customer_id BETWEEN 1 AND 5
ORDER BY customer_id

,customer_id,label
0,1,Aarav Nguyen (Chicago)
1,2,Priya Silva (Seattle)
2,3,John Lopez (Phoenix)
3,4,Maria Kim (Chicago)
4,5,Wei Khan (Chicago)


**DataFrame API:**

In [ ]:
res = (customers.filter(F.col("customer_id").between(1, 5))
       .select("customer_id",
               F.concat(F.col("name"), F.lit(" ("),
                        F.coalesce("city", F.lit("unknown")), F.lit(")")).alias("label"))
       .orderBy("customer_id"))
res.show()

+-----------+--------------------+
|customer_id|               label|
+-----------+--------------------+
|          1|Aarav Nguyen (Chi...|
|          2|Priya Silva (Seat...|
|          3|John Lopez (Phoenix)|
|          4| Maria Kim (Chicago)|
|          5|  Wei Khan (Chicago)|
+-----------+--------------------+

**Why / traps:** `||` joins text in all 3 engines. Careful: in BigQuery and Spark `'a' || NULL` is NULL, but Oracle treats NULL as an empty string here. That is why we use COALESCE.